# 03 — Validation: dNBR burned area vs NBAC perimeter

Burned = dNBR ≥ 0.10 (the FIREMON Low lower bound). Reference = the NBAC 2024 Jasper perimeter (NFIREID 451) rasterized to the 20 m grid; NBAC's unburned islands count as unburned.
Evaluation area = perimeter + 2 km buffer. Pixels inside **other** NBAC fires (2023–2025) are excluded, because they are real burns that aren't this fire.

Inputs: Drive `data/` from 01–02. Outputs: `figures/validation_metrics.csv`, `figures/validation_ndvi_sweep.csv`, `figures/agreement_map.png`

In [ ]:
![ -d /content/jasper-burn-severity ] || git clone https://github.com/MegaDeadCowboy/jasper-burn-severity.git /content/jasper-burn-severity
%cd /content/jasper-burn-severity
!pip install -q rioxarray geopandas

In [ ]:
import sys; sys.path.insert(0, ".")
from pathlib import Path
import urllib.request, zipfile
import numpy as np, pandas as pd, geopandas as gpd, rioxarray
from rasterio.features import rasterize
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from src.indices import ndvi
try:
    from google.colab import drive; drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/jasper-burn-severity/data")
except ImportError:
    DATA = Path("data")
FIG = Path("figures"); FIG.mkdir(exist_ok=True)

UTM, PIX_HA, BUFFER_M = 32611, 0.04, 2000
BURN_T = 0.10                                  # dNBR >= this = burned
FIRE_ID = 451                                  # Jasper fire NFIREID (from 01)
BANDS = ["B04", "B08", "B8A", "B12"]
for f in ["dnbr_2023_2025.tif", "s2_pre_2023.tif", "nbac_jasper_2024.gpkg"]:
    assert (DATA / f).exists(), f"missing {f}: run 01 and 02 first"

In [ ]:
d = rioxarray.open_rasterio(DATA / "dnbr_2023_2025.tif", masked=True).squeeze("band", drop=True)
pre = rioxarray.open_rasterio(DATA / "s2_pre_2023.tif", masked=True).assign_coords(band=BANDS).to_dataset("band")
ndvi_pre = ndvi(pre)
perim = gpd.read_file(DATA / "nbac_jasper_2024.gpkg").to_crs(UTM)
aoi = gpd.GeoDataFrame(geometry=perim.buffer(BUFFER_M), crs=UTM)
shape, transform = d.shape, d.rio.transform()
print(dict(d.sizes), d.rio.crs)

## 1. Other NBAC fires inside the evaluation area (2023–2025)

In [ ]:
NBAC_BASE = "https://cwfis.cfs.nrcan.gc.ca/downloads/nbac/NBAC_{yr}_20260513.zip"

def nbac_year(yr):
    z = DATA / f"NBAC_{yr}_20260513.zip"
    if not z.exists():
        urllib.request.urlretrieve(NBAC_BASE.format(yr=yr), z)
    out = Path(f"/content/nbac_{yr}") if Path("/content").exists() else DATA / f"nbac_{yr}"
    if not any(out.rglob("*.shp")):
        with zipfile.ZipFile(z) as zf: zf.extractall(out)
    return gpd.read_file(next(out.rglob("*.shp")), mask=aoi).to_crs(UTM)   # mask= reads only polygons touching the AOI

others = pd.concat([nbac_year(y) for y in (2023, 2024, 2025)], ignore_index=True)
others = others[~((others["YEAR"] == 2024) & (others["NFIREID"] == FIRE_ID))]
cols = [c for c in ["YEAR", "NFIREID", "AG_SDATE", "HS_SDATE", "POLY_HA"] if c in others.columns]
print(f"{len(others)} other NBAC polygons intersect the evaluation area")
others[cols].sort_values("POLY_HA", ascending=False)

## 2. Rasterize reference + masks

In [ ]:
def burn(geoms):
    geoms = [g for g in geoms if g is not None and not g.is_empty]
    if not geoms:
        return np.zeros(shape, bool)
    return rasterize(((g, 1) for g in geoms), out_shape=shape, transform=transform, fill=0, dtype="uint8").astype(bool)

ref = burn(perim.geometry)                      # NBAC burned (islands are holes -> False)
in_aoi = burn(aoi.geometry)
other_fire = burn(others.geometry) & ~ref       # keep overlap with the Jasper fire as reference
valid = in_aoi & ~other_fire & np.isfinite(d.values)
pred = d.values >= BURN_T
print(f"evaluation pixels: {valid.sum():,} | excluded as other fires: {(in_aoi & other_fire).sum():,} "
      f"({(in_aoi & other_fire).sum() * PIX_HA:,.1f} ha)")

## 3. Metrics

In [ ]:
def metrics(mask, label):
    p, r = pred[mask], ref[mask]
    tp, fp = int((p & r).sum()), int((p & ~r).sum())
    fn, tn = int((~p & r).sum()), int((~p & ~r).sum())
    return {"subset": label, "pixels": int(mask.sum()), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "IoU": tp / (tp + fp + fn), "precision": tp / (tp + fp), "recall": tp / (tp + fn),
            "F1": 2 * tp / (2 * tp + fp + fn), "overall_acc": (tp + tn) / mask.sum(),
            "pred_burned_ha": (tp + fp) * PIX_HA, "ref_burned_ha": (tp + fn) * PIX_HA,
            "area_diff_pct": 100 * (fp - fn) / (tp + fn)}

veg = ndvi_pre.values >= 0.30
results = pd.DataFrame([metrics(valid, "all pixels"),
                        metrics(valid & veg, "pre-fire NDVI >= 0.30")]).set_index("subset")
results.round(3).T

**Confusion matrix (all pixels)**: rows = NBAC, columns = dNBR.

In [ ]:
r0 = results.loc["all pixels"]
pd.DataFrame([[r0.TP, r0.FN], [r0.FP, r0.TN]],
             index=["NBAC burned", "NBAC unburned"], columns=["dNBR burned", "dNBR unburned"]).astype(int)

## 4. Sensitivity to a pre-fire vegetation mask
Non-vegetated pixels (rock, ice, water) can't burn, but their dNBR still moves between years. This sweep shows how much of the commission error they cause. **No cutoff is adopted here**: the chosen value goes in the Decision Log once it has been reviewed.

In [ ]:
sweep = pd.DataFrame([metrics(valid & (ndvi_pre.values >= t), f"NDVI >= {t:.2f}") | {"ndvi_cut": t}
                      for t in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]]).set_index("subset")
sweep[["pixels", "IoU", "precision", "recall", "F1", "area_diff_pct"]].round(3)

## 5. Agreement map

In [ ]:
# Categorical slots 1-3 of the dataviz reference palette (validated adjacent order, light mode)
AGREE = {"Both burned": "#2a78d6", "dNBR only (commission)": "#eb6834", "NBAC only (omission)": "#1baf7a"}
cls = np.full(shape, np.nan)
cls[valid & pred & ref] = 0
cls[valid & pred & ~ref] = 1
cls[valid & ~pred & ref] = 2
xs, ys = d.x.values, d.y.values
ext = [xs.min() - 10, xs.max() + 10, ys.min() - 10, ys.max() + 10]

base = pre["B8A"]; base = (base / base.quantile(0.98)).clip(0, 1)
fig, ax = plt.subplots(figsize=(8, 9), dpi=200)
ax.imshow(base, cmap="gray", vmin=0, vmax=1.4, extent=ext)
ax.imshow(cls, cmap=ListedColormap(list(AGREE.values())), vmin=-0.5, vmax=2.5, extent=ext, interpolation="nearest")
if len(others):
    others.boundary.plot(ax=ax, color="#7a7974", lw=0.5, linestyle="--")
perim.boundary.plot(ax=ax, color="#1a1a19", lw=0.6)
handles = [Patch(color=c, label=l) for l, c in AGREE.items()] + \
          [Patch(facecolor="none", edgecolor="#1a1a19", label="NBAC 2024 Jasper perimeter")]
if len(others):
    handles.append(Patch(facecolor="none", edgecolor="#7a7974", linestyle="--", label="Other NBAC fires 2023–25 (excluded)"))
ax.legend(handles=handles, loc="lower left", fontsize=8, framealpha=0.9,
          title=f"dNBR ≥ {BURN_T:.2f} vs NBAC  (IoU {r0.IoU:.2f}, all pixels)", title_fontsize=8)
ax.set_title("2024 Jasper Wildfire: burned-area agreement, Sentinel-2 dNBR vs NBAC", fontsize=9, loc="left")
ax.set_axis_off(); plt.tight_layout()
plt.savefig(FIG / "agreement_map.png", bbox_inches="tight")

## 6. Save

In [ ]:
results.round(4).to_csv(FIG / "validation_metrics.csv")
sweep.round(4).to_csv(FIG / "validation_ndvi_sweep.csv")
if len(others):
    others[cols].to_csv(FIG / "excluded_other_fires.csv", index=False)
print("saved")